In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

/Users/himanibhardwaj/himcodes/GenAI/LangGraph-Project/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
llm = ChatOllama(model='gemma3:270m')

In [3]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [4]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [7]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1)

{'topic': 'pizza',
 'joke': 'Why did the pizza get a bad review? \n\nBecause it was too big! \n',
 'explanation': "The joke plays on the common misconception that pizza is too large. It's a humorous twist on the classic pizza review.\n"}

In [8]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the pasta get fired? \n\nBecause it was too big!\n',
 'explanation': 'The joke plays on the common perception that pasta is large and therefore too big. The answer is that the pasta was too big to fit in the bowl. \n'}

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a bad review? \n\nBecause it was too big! \n', 'explanation': "The joke plays on the common misconception that pizza is too large. It's a humorous twist on the classic pizza review.\n"}, next=(), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f16fd40-b231-6508-8002-e190fe0a4617'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-06-24T13:53:13.579430+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f16fd40-ae96-6c18-8001-f5be16172523'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza get a bad review? \n\nBecause it was too big! \n'}, next=('generate_explanation',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f16fd40-ae96-6c18-8001-f5be16172523'}}, metadata={'source': 'loop', 'step': 1, 'parents': {}}, created_at='2026-06-2

In [11]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f16fd40-a512-6642-8000-daff0db0c032"}})

{'topic': 'pizza',
 'joke': 'Why did the pizza go to the doctor? \n\nBecause it was a bit too many! \n',
 'explanation': 'The joke plays on the common understanding of pizza being a large, often greasy, and potentially unhealthy food. The humor comes from the fact that the pizza is being described as "too many," implying that it\'s being eaten or consumed in a way that\'s excessive and potentially unhealthy. \n'}